# 🏛️ CA Vida — Demonstração Plataforma Snowflake
## Substituição SAS 9.4 → Snowflake | Workflow Solvency II

**Entidade**: CA Vida — Companhia de Seguros de Vida (Grupo Crédito Agrícola)  
**Cenário**: Processo de reporting regulatório Solvency II — já migrado para Snowflake

---
# ACT 1: Platform Foundation
> Compute elástico, ambientes isolados, recuperação instantânea e pipelines como código — tudo nativo, sem ferramentas externas.

## 1.1 — Arquitectura Multi-Ambiente
> Cada equipa tem compute independente. Se o ETL escalar, não afecta a produção. Escalar ou reduzir leva 2 segundos.

In [ ]:
SHOW WAREHOUSES LIKE 'CAVIDA%'

In [ ]:
SHOW SCHEMAS IN DATABASE CAVIDA_DEMO

In [ ]:
-- Escalar compute em 2 segundos
ALTER WAREHOUSE CAVIDA_PROD_WH SET WAREHOUSE_SIZE = 'MEDIUM';
SELECT 'PROD_WH escalado para MEDIUM' AS resultado;
ALTER WAREHOUSE CAVIDA_PROD_WH SET WAREHOUSE_SIZE = 'SMALL'

## 1.2 — Zero-Copy Clone
> Criar um ambiente de teste com TB de dados demora menos de 1 segundo e ocupa zero bytes adicionais de storage.

In [ ]:
CREATE OR REPLACE SCHEMA CAVIDA_DEMO.TEST CLONE CAVIDA_DEMO.PROD
  COMMENT = 'QA environment — zero-copy clone from PROD';

SELECT TABLE_SCHEMA, TABLE_NAME, ROW_COUNT
FROM CAVIDA_DEMO.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('PROD', 'TEST') AND TABLE_NAME LIKE 'SLV2%'
ORDER BY TABLE_SCHEMA, TABLE_NAME

## 1.3 — Time Travel (90 dias)
> Recuperação instantânea de dados eliminados ou alterados — até 90 dias de histórico automático, sem backups manuais.

In [ ]:
SELECT COUNT(*) AS registos_actuais FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

In [ ]:
-- Simular eliminação acidental
DELETE FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO WHERE asset_class = 'Cash';
SELECT COUNT(*) AS apos_delete FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

In [ ]:
-- Time Travel: ver dados ANTES da eliminação
SELECT COUNT(*) AS antes_da_eliminacao
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO AT(OFFSET => -60)

In [ ]:
-- Restaurar instantaneamente
INSERT INTO CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
SELECT * FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO AT(OFFSET => -60)
WHERE asset_class = 'Cash';
SELECT COUNT(*) AS restaurado FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

## 1.4 — dbt Project (Pipeline como Código)
> Pipeline de dados versionado, deployed como objecto Snowflake nativo. Estrutura: staging → intermediate → marts. Auditável, reproduzível, testável.

➡️ **Ver no Snowsight**: CAVIDA_DEMO → REGULATORY → dbt Projects → `CAVIDA_SLV2_PROJECT`

In [ ]:
SELECT table_type AS tipo, table_schema AS schema, table_name AS modelo
FROM CAVIDA_DEMO.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('DEV_STAGING','DEV_PROD')
ORDER BY table_schema, table_name

---
# ACT 2: Governance & Security
> Classificação automática, catálogo nativo, masking dinâmico e segurança por linha — tudo declarativo, sem licenças adicionais.

## 2.1 — Catálogo, Tags & Camada Semântica
> Dados classificados por domínio e sensibilidade. A Semantic View funciona como glossário de negócio — dimensões e métricas documentadas em português.

In [ ]:
SHOW TAGS IN SCHEMA CAVIDA_DEMO.REGULATORY

In [ ]:
SHOW SEMANTIC DIMENSIONS IN CAVIDA_DEMO.SEMANTIC.SLV2_ANALYTICS

In [ ]:
SHOW SEMANTIC METRICS IN CAVIDA_DEMO.SEMANTIC.SLV2_ANALYTICS

## 2.2 — EU Data Residency
> Dados alojados em Frankfurt (AWS EU-Central-1). Encriptação AES-256, edição Business Critical. Conformidade RGPD total.

In [ ]:
SELECT CURRENT_REGION() AS regiao, CURRENT_ACCOUNT_NAME() AS conta

## 2.3 — Dynamic Data Masking
> Mascaramento automático baseado no role do utilizador. Política declarativa aplicada em TODAS as queries — zero código manual.

In [ ]:
-- ACCOUNTADMIN vê dados completos
USE ROLE ACCOUNTADMIN;
SELECT instrument_name, counterparty, rating, market_value
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO LIMIT 5

In [ ]:
-- ANALYST vê dados MASCARADOS
USE ROLE CAVIDA_ANALYST;
USE WAREHOUSE CAVIDA_PROD_WH;
SELECT instrument_name, counterparty, rating, market_value
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO LIMIT 5

In [ ]:
USE ROLE ACCOUNTADMIN

## 2.4 — Row-Level Security
> Segurança invisível: o analista vê apenas dados dos últimos 6 meses sem sequer saber que existe mais. Sem erros, sem filtros visíveis.

In [ ]:
-- ACCOUNTADMIN vê TODOS os períodos
SELECT reference_date, COUNT(*) AS registos
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
GROUP BY 1 ORDER BY 1

In [ ]:
-- ANALYST só vê últimos 6 meses
USE ROLE CAVIDA_ANALYST;
USE WAREHOUSE CAVIDA_PROD_WH;
SELECT reference_date, COUNT(*) AS registos
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
GROUP BY 1 ORDER BY 1

In [ ]:
USE ROLE ACCOUNTADMIN

---
# ACT 3: Workflow Solvency II — Streamlit App
> Processo de reporting SLV II completo (16 passos) como web app nativa — write-back, upload de ficheiros, validações automáticas. Zero infra-estrutura externa.

➡️ **Abrir no Snowsight**: Streamlit → `CAVIDA_DEMO.REGULATORY.SLV2_WORKFLOW`

In [ ]:
SHOW STREAMLITS IN SCHEMA CAVIDA_DEMO.REGULATORY

---
# ACT 4: FinOps & Cost Intelligence
> Controlo total de custos com pay-per-second, resource monitors com alertas, e custo ZERO quando inactivo. Atribuição exacta por equipa/projecto.

In [ ]:
SHOW RESOURCE MONITORS LIKE 'CAVIDA%'

In [ ]:
SELECT
    warehouse_name AS warehouse,
    ROUND(SUM(credits_used), 2) AS creditos,
    ROUND(SUM(credits_used) * 3.50, 2) AS custo_eur
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE warehouse_name LIKE 'CAVIDA%'
  AND start_time >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY 1 ORDER BY 2 DESC

---
# ACT 5: Snowflake Intelligence — NLQ
> Consultas em linguagem natural sobre os dados SLV II — em português. O modelo gera SQL automaticamente e apresenta resultados com visualizações.

➡️ **Abrir no Snowsight**: Snowflake Intelligence → Agent `CA Vida - Solvency II Intelligence`

**Perguntas sugeridas:**
1. "Qual é o valor total da carteira de investimentos?"
2. "Mostra o balanço económico por tipo de activo"
3. "Qual é o capital disponível por tier?"
4. "Quantos instrumentos temos por classe de activo?"

In [ ]:
SHOW AGENTS IN SCHEMA CAVIDA_DEMO.SEMANTIC

---
# Resumo & Comparativo Final

## 20/20 Requisitos RFI Demonstrados ✅

| Área | Snowflake | Databricks | BigQuery |
|------|-----------|------------|----------|
| **Compute elástico & ambientes** | ✅ Warehouses isolados, auto-suspend, escala em 2s | ⚠️ Clusters always-on, resize lento | ✅ Slots auto, mas sem isolamento por equipa |
| **Zero-Copy Clone** | ✅ < 1 segundo, 0 bytes | ⚠️ Delta Clone (requer compute) | ❌ Não existe |
| **Time Travel** | ✅ 90 dias automáticos | ⚠️ Delta versioning (manual, VACUUM) | ⚠️ 7 dias máximo |
| **Pipeline como código** | ✅ dbt deployed nativo | ⚠️ dbt via Jobs API externo | ⚠️ Dataform (limitado) |
| **Catálogo & glossário** | ✅ Tags + Semantic View nativo | ⚠️ Unity Catalog (recente) | ⚠️ Data Catalog (separado) |
| **EU Data Residency** | ✅ Frankfurt, Business Critical | ✅ Azure/AWS EU disponível | ✅ EU disponível |
| **Dynamic Masking** | ✅ Declarativo, por role | ⚠️ Row/column filters Unity | ⚠️ Policy tags (limitado) |
| **Row-Level Security** | ✅ Invisível, declarativo | ⚠️ Row filters (menos maduro) | ⚠️ Row-level apenas IAM |
| **Web Apps nativas** | ✅ Streamlit in Snowflake | ❌ Não existe | ❌ Não existe |
| **Write-back** | ✅ UPDATE/INSERT directo | ❌ Não existe nativamente | ❌ Não existe nativamente |
| **FinOps** | ✅ Pay-per-second, monitors, zero idle | ❌ DBU caro, clusters ligados | ⚠️ Pay-per-query, sem monitors |
| **NLQ em português** | ✅ Snowflake Intelligence | ❌ Não existe nativamente | ❌ Não existe nativamente |
| **GenAI integrado** | ✅ Cortex (LLMs nativos) | ⚠️ Requer LangChain/externo | ⚠️ Requer Vertex AI |
| **Sector segurador** | ✅ Generali, AXA, Zurich, L&G | ⚠️ Menos referências | ⚠️ Menos referências |

## Próximos Passos

| # | Acção | Prazo |
|---|-------|-------|
| 1 | Workshop hands-on com equipa técnica | 2 semanas |
| 2 | PoC com dados reais SLV II | 4-6 semanas |
| 3 | Migração piloto de 1 pipeline SAS | 6-8 semanas |
| 4 | Plano de migração completo | 8-12 semanas |

---
*Snowflake — The AI Data Cloud*

# 🏛️ CA Vida — Demonstração Plataforma Snowflake
## Substituição SAS 9.4 → Snowflake | Workflow Solvency II

**Cliente**: CA Vida — Companhia de Seguros de Vida (Grupo Crédito Agrícola)  
**Duração**: 1h30min  
**Objectivo**: Demonstrar substituição completa do SAS 9.4 para reporting regulatório Solvency II  

---

| Snowflake ✅ | Databricks ❌ | BigQuery ❌ |
|---|---|---|
| Streamlit nativo (web apps) | Não tem app framework nativo | Não tem app framework nativo |
| Time Travel 90 dias | Delta Lake (gestão manual) | 7 dias máximo |
| Masking declarativo | Unity Catalog (recente, menos maduro) | Column-level apenas |
| Zero-Copy Clone instantâneo | Clone Delta (requer compute) | Não existe |
| NLQ nativo (Snowflake Intelligence) | Não existe nativamente | Não existe nativamente |
| dbt deploy nativo (objecto Snowflake) | Não existe | Não existe |
| Semantic View + Cortex Agent | Sem camada semântica nativa | Sem camada semântica nativa |
| Pay-per-second, auto-suspend | Always-on clusters (custo fixo) | Pay-per-query (sem controlo) |

---
# ACT 1: Platform Foundation (25 min)
## 1.1 — Arquitectura Multi-Ambiente

**💬 Mensagem**: "Já migrámos o vosso processo SAS para Snowflake. Separação total de compute e storage."

| Feature | Snowflake | SAS 9.4 | Databricks | BigQuery |
|---------|-----------|---------|------------|----------|
| Separação compute/storage | ✅ Nativa | ❌ Acoplados | ✅ Parcial | ✅ Nativa |
| Ambientes isolados | ✅ Schemas + WH | ❌ Servidores separados | ⚠️ Workspaces | ⚠️ Projects |
| Escalar sem mover dados | ✅ Instantâneo | ❌ Impossível | ⚠️ Resize | ✅ Automático |
| Custo idle | ✅ ZERO | ❌ 24/7 | ❌ Cluster activo | ✅ Zero |

In [ ]:
-- 4 warehouses dedicados por workload
SHOW WAREHOUSES LIKE 'CAVIDA%'

In [ ]:
-- Ambientes isolados por schema
SHOW SCHEMAS IN DATABASE CAVIDA_DEMO

In [ ]:
-- Escalar compute em 2 segundos — impossível no SAS!
ALTER WAREHOUSE CAVIDA_PROD_WH SET WAREHOUSE_SIZE = 'MEDIUM';
-- Verificar
SELECT 'PROD_WH escalado para MEDIUM em < 2 segundos' AS resultado;
-- Voltar ao tamanho original
ALTER WAREHOUSE CAVIDA_PROD_WH SET WAREHOUSE_SIZE = 'SMALL'

## 1.2 — Zero-Copy Clone (Instantâneo, 0 bytes extra)

**💬 Mensagem**: "Precisam de ambiente de teste? Instantâneo, sem duplicar dados."

| Feature | Snowflake | Databricks | BigQuery |
|---------|-----------|------------|----------|
| Clone instantâneo | ✅ Zero-Copy | ⚠️ Delta Clone (requer compute) | ❌ Não existe |
| Storage extra | 0 bytes até divergir | Duplicação parcial | N/A |
| Tempo para TB | < 1 segundo | Minutos a horas | N/A |

In [ ]:
-- Clone PROD para TEST — instantâneo, zero storage
CREATE OR REPLACE SCHEMA CAVIDA_DEMO.TEST CLONE CAVIDA_DEMO.PROD
  COMMENT = 'QA environment — zero-copy clone from PROD';
-- Verificar
SELECT TABLE_SCHEMA, TABLE_NAME, ROW_COUNT
FROM CAVIDA_DEMO.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('PROD', 'TEST') AND TABLE_NAME LIKE 'SLV2%'
ORDER BY TABLE_SCHEMA, TABLE_NAME

## 1.3 — Time Travel (90 dias, automático)

**💬 Mensagem**: "Se alguém apagar dados, recuperamos em segundos. Sem backup manual."

| Feature | Snowflake | Databricks | BigQuery |
|---------|-----------|------------|----------|
| Time Travel | ✅ 90 dias (BC) | ⚠️ Delta versions (manual) | ⚠️ 7 dias máx |
| UNDROP | ✅ Tabelas, schemas, DBs | ❌ Não existe | ❌ Não existe |
| Configuração | Zero (automático) | Requer VACUUM config | Automático mas limitado |

In [ ]:
-- Estado actual
SELECT COUNT(*) AS registos_actuais FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

In [ ]:
-- Simular eliminação acidental
DELETE FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO WHERE asset_class = 'Cash';
SELECT COUNT(*) AS apos_delete FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

In [ ]:
-- TIME TRAVEL! Ver dados ANTES da eliminação
SELECT COUNT(*) AS antes_da_eliminacao
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO AT(OFFSET => -60)

In [ ]:
-- RESTAURAR instantaneamente
INSERT INTO CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
SELECT * FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO AT(OFFSET => -60)
WHERE asset_class = 'Cash';
SELECT COUNT(*) AS restaurado FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

## 1.4 — dbt Project (Pipeline Deployed)

**💬 Mensagem**: "Pipeline versionado, deployed como objecto Snowflake nativo."

➡️ **Abrir no Snowsight**: CAVIDA_DEMO → REGULATORY → dbt Projects → `CAVIDA_SLV2_PROJECT`

| Camada | Modelos | Materialização |
|--------|---------|---------------|
| staging/ | stg_investment_portfolio, stg_liabilities, stg_risk_agility_inputs, stg_tagetik_data | Views |
| intermediate/ | int_asset_reconciliation, int_liability_reconciliation, int_risk_agility_processing | Views |
| marts/ | mart_economic_balance_sheet, mart_available_capital, mart_slv2_report | Tables |

In [ ]:
-- Objectos criados pelo dbt
SELECT table_type AS tipo, table_schema AS schema, table_name AS tabela
FROM CAVIDA_DEMO.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('DEV_STAGING','DEV_PROD')
ORDER BY table_schema, table_name

---
# ACT 2: Governance & Security (20 min)
## 2.1 — Catálogo, Tags & Semantic View

**💬 Mensagem**: "Todos os dados classificados. Sem licenças adicionais."

| Feature | Snowflake | Databricks | BigQuery |
|---------|-----------|------------|----------|
| Tags nativos | ✅ Ilimitados | ⚠️ Unity Catalog | ⚠️ Labels (limitados) |
| Semantic View | ✅ Glossário nativo | ❌ Não existe | ❌ Não existe |
| Propagação auto | ✅ Via lineage | ⚠️ Manual | ❌ Manual |

In [ ]:
-- Tags de governação aplicados
SHOW TAGS IN SCHEMA CAVIDA_DEMO.REGULATORY

In [ ]:
-- Semantic View = Glossário de Negócio (em português!)
SHOW SEMANTIC DIMENSIONS IN CAVIDA_DEMO.SEMANTIC.SLV2_ANALYTICS

In [ ]:
-- Métricas de negócio centralizadas
SHOW SEMANTIC METRICS IN CAVIDA_DEMO.SEMANTIC.SLV2_ANALYTICS

## 2.2 — EU Data Residency

**💬 Mensagem**: "Os vossos dados NUNCA saem da UE. Frankfurt, Business Critical, AES-256."

In [ ]:
-- Confirmar região UE
SELECT CURRENT_REGION() AS regiao, CURRENT_ACCOUNT_NAME() AS conta,
       'Business Critical' AS edicao, 'AES-256 + Tri-Secret Secure' AS encriptacao

## 2.3 — Dynamic Data Masking

**💬 Mensagem**: "Dados sensíveis mascarados automaticamente baseado no role. Zero código."

| Feature | Snowflake | Databricks | BigQuery |
|---------|-----------|------------|----------|
| Masking dinâmico | ✅ Por role, declarativo | ⚠️ Row/column filters | ⚠️ Policy tags |
| Aplicação | Automática TODAS queries | Manual por tabela | Manual por coluna |
| Flexibilidade | Full SQL expressions | Limitado | Limitado |

In [ ]:
-- Como ACCOUNTADMIN → vê dados completos
USE ROLE ACCOUNTADMIN;
SELECT instrument_name, counterparty, rating, market_value
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO LIMIT 5

In [ ]:
-- Como ANALYST → dados MASCARADOS automaticamente!
USE ROLE CAVIDA_ANALYST;
USE WAREHOUSE CAVIDA_PROD_WH;
SELECT instrument_name, counterparty, rating, market_value
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO LIMIT 5

In [ ]:
USE ROLE ACCOUNTADMIN

## 2.4 — Row-Level Security (RLS)

**💬 Mensagem**: "O analista só vê últimos 6 meses. Sem sequer saber que existe mais."

In [ ]:
-- ACCOUNTADMIN vê TODOS os períodos
SELECT reference_date, COUNT(*) AS registos
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
GROUP BY 1 ORDER BY 1

In [ ]:
-- ANALYST só vê últimos 6 meses — segurança INVISÍVEL!
USE ROLE CAVIDA_ANALYST;
USE WAREHOUSE CAVIDA_PROD_WH;
SELECT reference_date, COUNT(*) AS registos
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
GROUP BY 1 ORDER BY 1

In [ ]:
USE ROLE ACCOUNTADMIN

---
# ACT 3: Workflow Solvency II — Streamlit App (25 min)

**💬 Mensagem**: "O processo SLV II tal como o conhecem — mas dentro de Snowflake."

➡️ **Abrir**: Snowsight → Streamlit → `CAVIDA_DEMO.REGULATORY.SLV2_WORKFLOW`

### 16 Passos do Workflow:
| # | Passo | Highlight |
|---|-------|-----------|
| 1 | Período de Referência | **Write-back directo** (UPDATE) |
| 2 | Importação Carteira | Upload de ficheiros |
| 3 | Estado CA Gestl | Status colorido + progresso |
| 4 | Regras de Validação | 10 regras automáticas |
| 5 | Reconciliação Activos | Totais por classe |
| 6 | Reconciliação RA (Activos) | Registos vazios por ficheiro |
| 7 | Criação Ficheiros RA | Tabs + progresso |
| 8 | Reconciliação RA (Passivos) | CAPIT/FIXED/TAE |
| 9 | Download Ficheiros RA | Lista com download |
| 10 | Importação Output RA | Upload CSV |
| 11 | Estado Importação RA | Status colorido |
| 12 | Carregamento Tagetik | Pré-requisitos + progresso |
| 13 | Capital Disponível | Métricas por Tier (M€) |
| 14 | Balanço Económico | Variações + delta |
| 15 | Fecho Report | Botão de fecho |
| 16 | Download Tagetik | Ficheiros finais |

| Feature | Snowflake (Streamlit) | SAS | Databricks | BigQuery |
|---------|----------------------|-----|------------|----------|
| Web app nativa | ✅ | ❌ SAS VA (licença extra) | ❌ | ❌ |
| Write-back | ✅ UPDATE directo | ❌ CSV export/reimport | ❌ | ❌ |
| Infra adicional | Zero | Servidores SAS VA | Kubernetes | App Engine |
| Deploy | 1 comando | Semanas | N/A | N/A |

In [ ]:
-- Streamlit app deployed
SHOW STREAMLITS IN SCHEMA CAVIDA_DEMO.REGULATORY

---
# ACT 4: FinOps & Cost Intelligence (10 min)

**💬 Mensagem**: "Controlo total de custos. Impossível no modelo SAS."

| Feature | Snowflake | SAS | Databricks | BigQuery |
|---------|-----------|-----|------------|----------|
| Pay-per-use | ✅ Por segundo | ❌ Licença fixa | ⚠️ DBU (caro) | ✅ Por query |
| Resource Monitors | ✅ Alertas + suspend | ❌ Não existe | ❌ Budgets manual | ⚠️ Budgets |
| Attribution | ✅ Por warehouse/query | ❌ Impossível | ⚠️ Por cluster | ⚠️ Por projecto |
| Auto-suspend | ✅ 60s configurável | ❌ Sempre ligado | ❌ Manual | ✅ Automático |
| Custo noturno/fim-de-semana | ZERO | 100% | Cluster mínimo | Zero |

In [ ]:
-- 3 Resource Monitors com alertas e limites automáticos
SHOW RESOURCE MONITORS LIKE 'CAVIDA%'

In [ ]:
-- Custos por warehouse — atribuição exacta por equipa
SELECT
    warehouse_name AS warehouse,
    ROUND(SUM(credits_used), 2) AS creditos,
    ROUND(SUM(credits_used) * 3.50, 2) AS custo_eur
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE warehouse_name LIKE 'CAVIDA%'
  AND start_time >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY 1 ORDER BY 2 DESC

---
# ACT 5: WOW Factor — Snowflake Intelligence (10 min)

**💬 Mensagem**: "E agora, o futuro: perguntar aos dados em PORTUGUÊS."

➡️ **Abrir**: Snowflake Intelligence → Agent `CA Vida - Solvency II Intelligence`

### Perguntas para demonstrar:
1. "Qual é o valor total da carteira de investimentos?"
2. "Mostra o balanço económico por tipo de activo"
3. "Qual é o capital disponível por tier?"
4. "Quantos instrumentos temos por classe de activo?"

| Feature | Snowflake Intelligence | Databricks | BigQuery |
|---------|----------------------|------------|----------|
| NLQ nativo | ✅ Multi-língua (PT!) | ❌ Não existe | ❌ Não existe |
| Sobre dados internos | ✅ Sem mover dados | N/A | N/A |
| Visualização auto | ✅ Charts gerados | N/A | N/A |
| Cortex Agent | ✅ Orquestração LLM | ❌ Requer LangChain | ❌ Requer Vertex AI |

In [ ]:
-- Cortex Agent configurado
SHOW AGENTS IN SCHEMA CAVIDA_DEMO.SEMANTIC

---
# ✅ Resumo Final — 20/20 Requisitos

| Área | IDs | Snowflake | Databricks | BigQuery |
|------|-----|-----------|------------|----------|
| Architecture | 3,5,6 | ✅ Completo | ⚠️ Parcial | ⚠️ Parcial |
| Integration | 8,9,10 | ✅ + Streamlit | ❌ Sem apps | ❌ Sem apps |
| Governance | 13,14,16 | ✅ Nativo | ⚠️ Unity Catalog | ⚠️ Limitado |
| Security | 17,21 | ✅ Masking+RLS | ⚠️ Unity (recente) | ⚠️ Parcial |
| Operations | 30,32 | ✅ dbt nativo | ⚠️ Jobs externos | ⚠️ Dataform |
| FinOps | 34,35,36 | ✅ Pay-per-second | ❌ DBU caro | ⚠️ Pay-per-query |
| Semantic/AI | 37,42,46 | ✅ Intelligence | ❌ Sem NLQ | ❌ Sem NLQ |
| Adoption | 47 | ✅ SLV II completo | - | - |

## Próximos Passos

| # | Acção | Prazo |
|---|-------|-------|
| 1 | Workshop hands-on com equipa técnica | 2 semanas |
| 2 | PoC com dados reais SLV II | 4-6 semanas |
| 3 | Migração piloto de 1 pipeline SAS | 6-8 semanas |
| 4 | Plano de migração completo | 8-12 semanas |

---
*Snowflake — The AI Data Cloud | CA Vida Demo | Maio 2026*